# 🔄 Day 04a: REST & WebSocket — APIs and Real-Time Communication

---

## 🎯 What You'll Master Today
- REST constraints and RESTful API design
- WebSocket vs HTTP
- Server-Sent Events, Long Polling
- gRPC basics

---

```
╔══════════════════════════════════════════════════════════════════════╗
║         COMMUNICATION PATTERNS                                       ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  1. HTTP (Request/Response)                                          ║
║     Client ──request──→ Server ──response──→ Client (done)         ║
║                                                                      ║
║  2. Long Polling                                                     ║
║     Client ──request──→ Server [holds...holds...] ──response──→    ║
║     Client ──request──→ Server [holds...] ──response──→ (repeat)   ║
║                                                                      ║
║  3. Server-Sent Events (SSE)                                        ║
║     Client ──request──→ Server ──event──→ ──event──→ ──event──→   ║
║     (one-way stream: server → client only)                          ║
║                                                                      ║
║  4. WebSocket                                                        ║
║     Client ←──────────→ Server  (full-duplex, bidirectional)       ║
║     (persistent connection, both sides send anytime)                ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. REST Constraints (Roy Fielding's 6 Rules)
# ============================================

rest_constraints = [
    ("Client-Server",
     "Separation of concerns. Client handles UI, server handles data.",
     "Mobile, web, CLI can all use the same API"),
    ("Stateless",
     "Each request contains ALL info needed. No server-side session.",
     "Scalable — any server can handle any request"),
    ("Cacheable",
     "Responses must declare if cacheable (Cache-Control, ETag).",
     "Reduces server load, improves latency"),
    ("Uniform Interface",
     "Resource identification (URI), self-descriptive messages, HATEOAS.",
     "Consistent API design across all endpoints"),
    ("Layered System",
     "Client doesn't know if talking to server or proxy/LB.",
     "CDN, load balancer, API gateway — transparent"),
    ("Code on Demand",
     "Server can send executable code (JavaScript). OPTIONAL.",
     "Rarely used in APIs, more in web pages"),
]

print("  ═══ 6 REST Constraints (Roy Fielding, 2000) ═══\n")
for i, (name, desc, why) in enumerate(rest_constraints, 1):
    optional = " (optional)" if i == 6 else ""
    print(f"  {i}. {name}{optional}")
    print(f"     {desc}")
    print(f"     Why: {why}\n")

In [ ]:
# ============================================
# 2. RESTful API Design Best Practices
# ============================================

print("""
  ═══ RESTful URL Design ═══

  ✅ Good (nouns, plural, hierarchical):
    GET    /api/v1/users              → List users
    POST   /api/v1/users              → Create user
    GET    /api/v1/users/123          → Get user 123
    PUT    /api/v1/users/123          → Update user 123
    DELETE /api/v1/users/123          → Delete user 123
    GET    /api/v1/users/123/orders   → User 123's orders

  ❌ Bad (verbs in URL, inconsistent):
    GET    /api/getUsers
    POST   /api/createUser
    GET    /api/user/delete/123

  Query params for filtering/pagination:
    GET /api/v1/users?status=active&sort=name&page=2&limit=20

  Versioning strategies:
    URL:    /api/v1/users     (most common)
    Header: Accept: application/vnd.api+json;version=1
    Query:  /api/users?version=1
""")

In [ ]:
# ============================================
# 3. WebSocket Simulation
# ============================================

class WebSocketSimulator:
    def __init__(self):
        self.connected = False
        self.messages = []
    
    def handshake(self):
        print("  ═══ WebSocket Handshake (HTTP Upgrade) ═══\n")
        print("  Client → Server:")
        print("    GET /chat HTTP/1.1")
        print("    Host: example.com")
        print("    Upgrade: websocket")
        print("    Connection: Upgrade")
        print("    Sec-WebSocket-Key: dGhlIHNhbXBsZSBub25jZQ==")
        print("    Sec-WebSocket-Version: 13")
        print()
        print("  Server → Client:")
        print("    HTTP/1.1 101 Switching Protocols")
        print("    Upgrade: websocket")
        print("    Connection: Upgrade")
        print("    Sec-WebSocket-Accept: s3pPLMBiTxaQ9kYGzzhZRbK+xOo=")
        self.connected = True
        print("\n  ✅ WebSocket connection established!")
        print("  Now both sides can send messages anytime.\n")
    
    def send(self, sender, message):
        if not self.connected:
            print("  ❌ Not connected!")
            return
        self.messages.append((sender, message))
        print(f"    {sender}: {message}")

ws = WebSocketSimulator()
ws.handshake()

print("  ═══ Bidirectional Communication ═══")
ws.send("Client", "Hello server!")
ws.send("Server", "Hello client! You have 3 notifications.")
ws.send("Server", "New message from Alice")
ws.send("Client", "Read notification 1")
ws.send("Server", "New message from Bob")

print(f"\n  Total messages exchanged: {len(ws.messages)}")
print("  Key: server can PUSH without client asking!")

In [ ]:
# ============================================
# 4. Communication Pattern Comparison
# ============================================

print("""
  ┌────────────────┬──────────┬──────────────┬───────────┬──────────────────┐
  │ Pattern         │ Direction│ Connection   │ Overhead  │ Use Case         │
  ├────────────────┼──────────┼──────────────┼───────────┼──────────────────┤
  │ HTTP polling    │ Client→  │ New each time│ Very High │ Simple updates   │
  │ Long polling    │ Client→  │ Held open    │ High      │ Near-realtime    │
  │ SSE             │ Server→  │ Persistent   │ Low       │ Notifications    │
  │ WebSocket       │ Both ↔   │ Persistent   │ Very Low  │ Chat, gaming     │
  │ gRPC            │ Both ↔   │ HTTP/2       │ Very Low  │ Microservices    │
  └────────────────┴──────────┴──────────────┴───────────┴──────────────────┘

  When to use what:
  ┌────────────────────────────────────────────────────────────────────┐
  │ Need real-time + bidirectional?        → WebSocket               │
  │ Server pushes events (one-way)?        → SSE                     │
  │ Simple, infrequent updates?            → HTTP polling            │
  │ Can't use WebSocket (proxy issues)?    → Long polling            │
  │ Microservice-to-microservice?          → gRPC                    │
  │ Browser client + REST API?             → HTTP (the default)      │
  └────────────────────────────────────────────────────────────────────┘

  gRPC vs REST:
    gRPC: Protocol Buffers (binary), HTTP/2, streaming, code-gen
    REST: JSON (text), HTTP/1.1+, request/response, human-readable
    gRPC is 2-10x faster but less browser-friendly
""")

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | What makes an API RESTful? | 6 constraints: client-server, stateless, cacheable, uniform interface, layered, code-on-demand |
| 2 | WebSocket vs HTTP? | WS: persistent, bidirectional, low overhead. HTTP: request/response, stateless |
| 3 | When to use WebSocket vs SSE? | WS: bidirectional (chat). SSE: server→client only (notifications, live scores) |
| 4 | What is long polling? | Client sends request, server holds until data available, responds, client immediately re-requests |
| 5 | gRPC vs REST? | gRPC: binary (protobuf), HTTP/2, streaming, fast. REST: JSON, HTTP/1.1, human-readable |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • REST: stateless, resource-based, uniform interface    │
## │  • WebSocket: persistent, bidirectional via HTTP upgrade │
## │  • SSE: server→client one-way streaming over HTTP       │
## │  • gRPC: binary, streaming, fast micro-service comms    │
## │  • Default to REST. Use WS/SSE when you need real-time  │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **TCP** — Reliable data delivery with 3-way handshake